<a href="https://colab.research.google.com/github/balloon083/food-recognition-cnn/blob/main/food_recognition_pipeline_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q gdown opencv-python-headless torch torchvision timm

import os, json, random, time, math, glob, tarfile
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision import models
from collections import Counter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected. Go to Runtime > Change runtime type > GPU before continuing.")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

Using device: cuda
GPU: Tesla T4


## 2. Download and extract Khana

Distributed as a `.tar.gz` on Google Drive (~6.4 GB). If this link ever goes stale, get the current one
from https://khana.omkarprabhu.in/

In [ ]:
DATA_ROOT = "/content/khana"
os.makedirs(DATA_ROOT, exist_ok=True)

KHANA_DRIVE_FOLDER = "https://drive.google.com/drive/folders/1PWyJdkizw5ABBd8BIAnr_FZq91YZ2Uo0"
!gdown --folder "$KHANA_DRIVE_FOLDER" -O "$DATA_ROOT"

# It comes down as dataset.tar.gz alongside labels.txt and taxonomy.csv
tar_path = os.path.join(DATA_ROOT, "dataset.tar.gz")
with tarfile.open(tar_path, "r:gz") as tar:
    tar.extractall(DATA_ROOT)

# Extraction nests everything one level deeper
IMAGE_ROOT = os.path.join(DATA_ROOT, "khana")
print("Extracted. Top-level entries under IMAGE_ROOT:")
print(sorted(os.listdir(IMAGE_ROOT))[:10])

## 3. Build the class list and scan every image (filtering known junk)

In [ ]:
all_entries = sorted(os.listdir(IMAGE_ROOT))
class_names = sorted([
    e for e in all_entries
    if os.path.isdir(os.path.join(IMAGE_ROOT, e))
])
print(f"{len(class_names)} dish classes found")  # expect 80

class_to_idx = {name: i for i, name in enumerate(class_names)}
idx_to_class = {i: name for name, i in class_to_idx.items()}
NUM_CLASSES = len(class_names)

In [ ]:
JUNK_PREFIXES = ("._",)
JUNK_NAMES = {".DS_Store"}

all_samples = []
for cname in class_names:
    cdir = os.path.join(IMAGE_ROOT, cname)
    for fname in os.listdir(cdir):
        if fname in JUNK_NAMES or fname.startswith(JUNK_PREFIXES):
            continue
        fpath = os.path.join(cdir, fname)
        if os.path.isfile(fpath):
            all_samples.append((fpath, cname))

print(f"Total real images found: {len(all_samples)}")  # expect ~131,819

# Sanity check: confirm no unreadable files slipped through (this takes a few minutes)
print("Verifying every file is readable (this takes a few minutes)...")
bad = [p for p, _ in all_samples if cv2.imread(p) is None]
print(f"Unreadable files: {len(bad)}")
if bad:
    print("First few bad paths:", bad[:5])
    print("Removing them from the sample list before continuing.")
    bad_set = set(bad)
    all_samples = [(p, c) for p, c in all_samples if p not in bad_set]
    print(f"Remaining samples: {len(all_samples)}")

**Known limitation — class imbalance.** Some dishes have as few as ~125 images, others have
close to 12,000. This run does **not** correct for it (no class weighting or resampling) — this is a
conscious scope decision for a presentation build, not an oversight. If you extend this later, add
`weight=...` to the loss function using inverse class frequency.

In [ ]:
class_counts = Counter(label for _, label in all_samples)
sorted_counts = sorted(class_counts.items(), key=lambda x: x[1])
print("5 smallest classes:", sorted_counts[:5])
print("5 largest classes:", sorted_counts[-5:])

## 4. Train / Val / Test split (80 / 10 / 10)

In [ ]:
random.seed(SEED)
random.shuffle(all_samples)

n = len(all_samples)
n_train = int(0.8 * n)
n_val = int(0.1 * n)

train_samples = all_samples[:n_train]
val_samples = all_samples[n_train:n_train + n_val]
test_samples = all_samples[n_train + n_val:]

print(f"Train: {len(train_samples)}  Val: {len(val_samples)}  Test: {len(test_samples)}")

## 5. OpenCV Preprocessing

CLAHE lighting normalization (in LAB space, so color is untouched) + Otsu-threshold contour
detection to crop to the food region + resize. Runs identically at train and inference time.

In [ ]:
def preprocess_food_image(image_bgr, target_size=224):
    img = image_bgr.copy()

    # Lighting normalization
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    lab = cv2.merge((l, a, b))
    img = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

    # Portion framing: crop to the largest detected region
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        largest = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(largest)
        pad = int(0.05 * max(w, h))
        x0, y0 = max(x - pad, 0), max(y - pad, 0)
        x1, y1 = min(x + w + pad, img.shape[1]), min(y + h + pad, img.shape[0])
        if (x1 - x0) > 20 and (y1 - y0) > 20:
            img = img[y0:y1, x0:x1]

    img = cv2.resize(img, (target_size, target_size), interpolation=cv2.INTER_AREA)
    return img

In [ ]:
# Quick visual check
import matplotlib.pyplot as plt

sample_path = train_samples[0][0]
raw = cv2.imread(sample_path)
processed = preprocess_food_image(raw)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)); axes[0].set_title("Raw"); axes[0].axis("off")
axes[1].imshow(cv2.cvtColor(processed, cv2.COLOR_BGR2RGB)); axes[1].set_title("Preprocessed"); axes[1].axis("off")
plt.show()

## 6. Dataset and DataLoaders

In [ ]:
class FoodDataset(Dataset):
    def __init__(self, samples, class_to_idx, train=True, img_size=224):
        self.samples = samples
        self.class_to_idx = class_to_idx
        self.img_size = img_size
        self.train = train

        if train:
            self.augment = T.Compose([
                T.ToPILImage(),
                T.RandomHorizontalFlip(),
                T.RandomRotation(15),
                T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
                T.ToTensor(),
                T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ])
        else:
            self.augment = T.Compose([
                T.ToPILImage(),
                T.ToTensor(),
                T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, class_name = self.samples[idx]
        img_bgr = cv2.imread(path)
        if img_bgr is None:
            img_bgr = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
        img_bgr = preprocess_food_image(img_bgr, target_size=self.img_size)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        tensor = self.augment(img_rgb)
        label = self.class_to_idx[class_name]
        return tensor, label

In [ ]:
BATCH_SIZE = 64
IMG_SIZE = 224
NUM_WORKERS = 2  # this machine only has 2 CPU cores — using more causes warnings/slowdowns

train_ds = FoodDataset(train_samples, class_to_idx, train=True, img_size=IMG_SIZE)
val_ds = FoodDataset(val_samples, class_to_idx, train=False, img_size=IMG_SIZE)
test_ds = FoodDataset(test_samples, class_to_idx, train=False, img_size=IMG_SIZE)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# Sanity check before committing to a full run
images, labels = next(iter(train_loader))
print("Batch of images shape:", images.shape)   # expect [64, 3, 224, 224]
print("Batch of labels shape:", labels.shape)   # expect [64]

## 7. Model — EfficientNet-B0 transfer learning

Pretrained on ImageNet, classifier head replaced for our 80 classes. Low-level visual features
(edges, textures, color gradients) transfer well from ImageNet even though its 1000 categories
aren't food-specific — those features are general-purpose, not "dog-specific" or "car-specific."

In [ ]:
def build_model(num_classes):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features, num_classes),
    )
    return model.to(device)

model = build_model(NUM_CLASSES)
print(model.classifier)

## 8. Training Loop

In [ ]:
def topk_accuracy(output, target, ks=(1, 5)):
    maxk = max(ks)
    batch_size = target.size(0)
    _, pred = output.topk(maxk, dim=1, largest=True, sorted=True)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))
    return [(correct[:k].reshape(-1).float().sum(0, keepdim=True) / batch_size).item() for k in ks]


def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, total_top1, total_top5, n_batches = 0.0, 0.0, 0.0, 0

    with torch.set_grad_enabled(is_train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if is_train:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            if is_train:
                loss.backward()
                optimizer.step()
            top1, top5 = topk_accuracy(outputs, labels, ks=(1, 5))
            total_loss += loss.item(); total_top1 += top1; total_top5 += top5; n_batches += 1

    return total_loss / n_batches, total_top1 / n_batches, total_top5 / n_batches

In [ ]:
NUM_EPOCHS = 15
LR = 3e-4

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

best_val_top1 = 0.0
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    train_loss, train_top1, train_top5 = run_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_top1, val_top5 = run_epoch(model, val_loader, criterion, optimizer=None)
    scheduler.step()

    history.append({"epoch": epoch, "train_loss": train_loss, "train_top1": train_top1,
                     "train_top5": train_top5, "val_loss": val_loss, "val_top1": val_top1, "val_top5": val_top5})

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | train loss {train_loss:.3f} top1 {train_top1:.3f} top5 {train_top5:.3f} "
          f"| val loss {val_loss:.3f} top1 {val_top1:.3f} top5 {val_top5:.3f} | {time.time()-t0:.0f}s")

    if val_top1 > best_val_top1:
        best_val_top1 = val_top1
        torch.save({"model_state": model.state_dict(), "class_to_idx": class_to_idx, "img_size": IMG_SIZE},
                   "/content/best_food_model.pt")
        print(f"  new best model saved (val top1 {val_top1:.3f})")

**If you're short on time before the presentation**: you don't need all 15 epochs to finish.
Once val top-1 accuracy plateaus (stops improving for 2-3 epochs in a row), you can interrupt the cell —
the *best* checkpoint so far is already saved to `/content/best_food_model.pt` on every improvement, so
stopping early doesn't lose your best result.

## 9. Final Evaluation on Held-Out Test Set

In [ ]:
checkpoint = torch.load("/content/best_food_model.pt", map_location=device)
model.load_state_dict(checkpoint["model_state"])

test_loss, test_top1, test_top5 = run_epoch(model, test_loader, criterion, optimizer=None)
print(f"Test top-1 accuracy: {test_top1*100:.2f}%")
print(f"Test top-5 accuracy: {test_top5*100:.2f}%")
print()
print("For reference, Khana's own published ConvNeXt-S baseline: 86.72% top-1 / 97.58% top-5")

## 10. Inference Latency Benchmark

Your resume claims sub-180ms end-to-end latency. Let's actually measure it rather than assume it,
including preprocessing time (OpenCV runs on CPU) plus model inference.

In [ ]:
def benchmark_inference(model, sample_path, n_runs=20):
    times = []
    for _ in range(n_runs):
        t0 = time.time()

        img_bgr = cv2.imread(sample_path)
        processed = preprocess_food_image(img_bgr, target_size=IMG_SIZE)
        rgb = cv2.cvtColor(processed, cv2.COLOR_BGR2RGB)
        transform = T.Compose([
            T.ToPILImage(), T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
        tensor = transform(rgb).unsqueeze(0).to(device)

        model.eval()
        with torch.no_grad():
            _ = model(tensor)

        times.append((time.time() - t0) * 1000)  # ms

    return np.mean(times), np.median(times), np.min(times), np.max(times)


sample_path = test_samples[0][0]
mean_ms, median_ms, min_ms, max_ms = benchmark_inference(model, sample_path)
print(f"Mean latency: {mean_ms:.1f}ms | Median: {median_ms:.1f}ms | Min: {min_ms:.1f}ms | Max: {max_ms:.1f}ms")
print()
print("Note: this is measured on Colab's GPU + shared CPU. On-device (phone) latency for the actual")
print("app will differ — this number tells you whether the *approach* is fast enough, not the final")
print("production number.")

## 11. Calorie Estimation (heuristic — not validated against real data)

**Read this before presenting the MAPE number on your resume.** This dataset has no real calorie
ground truth. The table below is illustrative, not sourced from a nutrition database, and only
covers 5 of your 80 classes as a placeholder. There is no honest way to report a 16% MAPE without
real calorie-annotated test data to measure against — if asked, say this part is a heuristic
built and ready for real data, not a validated number yet.

In [ ]:
CALORIES_PER_SERVING = {
    "biryani": 450,
    "masala dosa": 168,
    "naan": 260,  # only include if this exact class name exists in class_names — check below
    "dal": 200,
    "jalebi": 150,
}

# Confirm which of these placeholder names actually exist in this dataset's 80 classes
for name in CALORIES_PER_SERVING:
    print(name, "-> in dataset:", name in class_to_idx)

DEFAULT_CALORIES = 300

def estimate_portion_scale(food_region_area, frame_area):
    baseline_occupancy = 0.35
    occupancy = food_region_area / frame_area
    scale = occupancy / baseline_occupancy
    return max(0.4, min(scale, 2.5))

def estimate_calories(class_name, confidence, food_region_area, frame_area):
    base = CALORIES_PER_SERVING.get(class_name, DEFAULT_CALORIES)
    portion_scale = estimate_portion_scale(food_region_area, frame_area)
    avg_calories = np.mean(list(CALORIES_PER_SERVING.values())) if CALORIES_PER_SERVING else DEFAULT_CALORIES
    confidence_adjusted = confidence * base + (1 - confidence) * avg_calories
    return confidence_adjusted * portion_scale

## 12. End-to-End Inference

In [ ]:
def predict_calories(image_path, model, class_to_idx, idx_to_class, img_size=224):
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise ValueError(f"Could not read image: {image_path}")

    frame_area = img_bgr.shape[0] * img_bgr.shape[1]

    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    food_region_area = cv2.contourArea(max(contours, key=cv2.contourArea)) if contours else frame_area * 0.35

    processed = preprocess_food_image(img_bgr, target_size=img_size)
    rgb = cv2.cvtColor(processed, cv2.COLOR_BGR2RGB)
    transform = T.Compose([
        T.ToPILImage(), T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    tensor = transform(rgb).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1)
        confidence, pred_idx = probs.max(dim=1)

    class_name = idx_to_class[pred_idx.item()]
    confidence = confidence.item()
    calorie_estimate = estimate_calories(class_name, confidence, food_region_area, frame_area)

    return {"predicted_dish": class_name, "confidence": round(confidence, 3),
            "estimated_calories": round(calorie_estimate)}


# Try it on a real test image
demo_path = test_samples[0][0]
print("True label:", test_samples[0][1])
print(predict_calories(demo_path, model, class_to_idx, idx_to_class))

## For tomorrow's presentation — honest talking points

- **Dataset**: Khana, 131,000+ images, **80 dish classes** (not 100+ — say this plainly if asked;
  the path to 100+ is merging a second dataset, which is a known next step, not done yet)
- **Accuracy**: whatever your actual training run prints in Section 9 — compare it against Khana's
  own published baseline (86.72% top-1 / 97.58% top-5) as a reference point, not a promise
- **Latency**: whatever Section 10 actually measures on this hardware — be clear it's measured on
  Colab, not the final on-device target
- **Calorie estimation**: describe it as a working heuristic pipeline (confidence + portion-size
  scaling), explicitly note there's no real calorie ground truth yet to report a validated error
  rate against — don't present 16% MAPE as measured, since it isn't
- **Known limitation, stated proactively**: severe class imbalance (94x between largest and
  smallest class) wasn't corrected in this build — a deliberate scope decision for a presentation
  timeline, not something you're unaware of